In [43]:
from __future__ import annotations

import csv
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

import shutil

import pandas as pd

In [37]:
ROOT_DIRECTORY = Path("/Volumes/LaCie/DAIC-WOZ/Dataset")

# Processed recordings will be saved under this directory
PROCESSED_DIRECTORY = Path("/Volumes/LaCie/DAIC-WOZ/Processed Dataset")

INTERVIEWER_NAME = "Ellie"

TRANSCRIPT_SUFFIX = "_TRANSCRIPT.csv"
AUDIO_SUFFIX = "_AUDIO.wav"

# False prevents existing processed files from being replaced
OVERWRITE_EXISTING_FILES = False

In [38]:
def read_removal_intervals(transcript_path: Path, interviewer_name: str = INTERVIEWER_NAME) -> list[tuple[float, float]]:
    """
    Read a tab-separated transcript and determine which audio intervals
    should be removed.

    Removal rules
    -------------
    1. Remove audio from 0.0 seconds to the start time of the first
       valid transcript row, regardless of the speaker.

    2. Remove every interval where Ellie is speaking.

    3. When two or more consecutive transcript rows belong to Ellie,
       merge them into one interval. This also removes any pause between
       those consecutive Ellie rows.

    4. A row from another speaker ends the current Ellie interval.

    Expected transcript columns
    ---------------------------
    start_time
    stop_time
    speaker
    value

    Parameters
    ----------
    transcript_path:
        Path to the tab-separated _TRANSCRIPT.csv file.

    interviewer_name:
        Speaker name used for the interviewer.

    Returns
    -------
    list[tuple[float, float]]
        Start and stop times, in seconds, that should be removed.
    """
    intervals_to_remove: list[tuple[float, float]] = []

    current_ellie_start: float | None = None
    current_ellie_stop: float | None = None

    previous_speaker: str | None = None
    first_valid_row_found = False

    with transcript_path.open(
        mode="r",
        encoding="utf-8-sig",
        newline="",
    ) as transcript_file:
        reader = csv.DictReader(
            transcript_file,
            delimiter="\t",
        )

        if reader.fieldnames is None:
            raise ValueError(
                f"No header was found in transcript: {transcript_path}"
            )

        # Remove accidental whitespace from transcript column names.
        reader.fieldnames = [
            column.strip()
            for column in reader.fieldnames
        ]

        required_columns = {
            "start_time",
            "stop_time",
            "speaker",
        }

        missing_columns = required_columns.difference(
            reader.fieldnames
        )

        if missing_columns:
            raise ValueError(
                f"{transcript_path.name} is missing required columns: "
                f"{sorted(missing_columns)}. "
                f"Found columns: {reader.fieldnames}"
            )

        for row_number, row in enumerate(reader, start=2):
            speaker_value = row.get("speaker")
            start_value = row.get("start_time")
            stop_value = row.get("stop_time")

            if (
                speaker_value is None
                or start_value is None
                or stop_value is None
            ):
                print(
                    f"Warning: missing transcript data in "
                    f"{transcript_path.name}, row {row_number}. "
                    "Skipping row."
                )
                continue

            speaker = speaker_value.strip()

            try:
                start_time = float(start_value)
                stop_time = float(stop_value)

            except (TypeError, ValueError):
                print(
                    f"Warning: invalid timestamp in "
                    f"{transcript_path.name}, row {row_number}. "
                    "Skipping row."
                )
                continue

            start_time = max(0.0, start_time)

            if stop_time <= start_time:
                print(
                    f"Warning: stop_time must be greater than start_time "
                    f"in {transcript_path.name}, row {row_number}. "
                    "Skipping row."
                )
                continue

            # Remove everything before the first valid transcript row,
            # regardless of whether Ellie or the participant speaks first.
            if not first_valid_row_found:
                if start_time > 0.0:
                    intervals_to_remove.append(
                        (0.0, start_time)
                    )

                first_valid_row_found = True

            if speaker == interviewer_name:
                if (
                    previous_speaker == interviewer_name
                    and current_ellie_start is not None
                    and current_ellie_stop is not None
                ):
                    # The previous and current rows are both Ellie.
                    # Extend the current removal interval through the
                    # current row's stop time.
                    current_ellie_stop = max(
                        current_ellie_stop,
                        stop_time,
                    )

                else:
                    # Start a new Ellie interval.
                    current_ellie_start = start_time
                    current_ellie_stop = stop_time

            else:
                # A non-Ellie row ends the current Ellie sequence.
                if (
                    previous_speaker == interviewer_name
                    and current_ellie_start is not None
                    and current_ellie_stop is not None
                ):
                    intervals_to_remove.append(
                        (
                            current_ellie_start,
                            current_ellie_stop,
                        )
                    )

                current_ellie_start = None
                current_ellie_stop = None

            previous_speaker = speaker

    # Save the final Ellie interval when the transcript ends with Ellie.
    if (
        previous_speaker == interviewer_name
        and current_ellie_start is not None
        and current_ellie_stop is not None
    ):
        intervals_to_remove.append(
            (
                current_ellie_start,
                current_ellie_stop,
            )
        )

    if not first_valid_row_found:
        raise ValueError(
            f"No valid transcript rows were found in: {transcript_path}"
        )

    return intervals_to_remove

In [22]:
def merge_removal_intervals(intervals: list[tuple[float, float]]) -> list[tuple[float, float]]:
    """
    Merge overlapping or directly adjacent removal intervals.

    Example
    -------
    [
        (0.0, 29.0),
        (29.0, 35.0),
        (40.0, 43.0),
    ]

    becomes:

    [
        (0.0, 35.0),
        (40.0, 43.0),
    ]

    This function does not merge intervals separated by a positive gap.
    Consecutive Ellie rows have already been merged while parsing.
    """
    if not intervals:
        return []

    sorted_intervals = sorted(intervals, key=lambda interval: interval[0])

    merged: list[list[float]] = [[sorted_intervals[0][0], sorted_intervals[0][1]]]

    for start_time, stop_time in sorted_intervals[1:]:
        previous_stop = merged[-1][1]

        if start_time <= previous_stop:
            merged[-1][1] = max(previous_stop, stop_time)
        else:
            merged.append([start_time, stop_time])

    return [
        (start_time, stop_time)
        for start_time, stop_time in merged
    ]

In [23]:
def remove_audio_intervals(audio: np.ndarray, sample_rate: int, intervals_to_remove: list[tuple[float, float]]) -> np.ndarray:
    """
    Remove specified intervals from an audio array.

    The function does not repeatedly delete from the audio array.
    Instead, it gathers the sections that should be retained from the
    original audio and concatenates those sections once.

    Parameters
    ----------
    audio:
        Audio loaded by librosa.

        Mono shape:
            (samples,)

        Multichannel shape:
            (channels, samples)

    sample_rate:
        Number of audio samples per second.

    intervals_to_remove:
        Start and stop times, in seconds, to remove.

    Returns
    -------
    np.ndarray
        Audio with the specified intervals removed.
    """
    if sample_rate <= 0:
        raise ValueError(
            f"sample_rate must be positive. Received: {sample_rate}"
        )

    total_samples = audio.shape[-1]
    audio_duration = total_samples / sample_rate

    valid_intervals: list[tuple[float, float]] = []

    for start_time, stop_time in intervals_to_remove:
        clipped_start = max(0.0, min(start_time, audio_duration))

        clipped_stop = max(0.0, min(stop_time, audio_duration))

        if clipped_stop > clipped_start:
            valid_intervals.append((clipped_start, clipped_stop))

    valid_intervals = merge_removal_intervals(valid_intervals)

    if not valid_intervals:
        return audio.copy()

    retained_segments: list[np.ndarray] = []
    previous_stop_sample = 0

    for start_time, stop_time in valid_intervals:
        start_sample = int(round(start_time * sample_rate))

        stop_sample = int(round(stop_time * sample_rate))

        start_sample = max(0, min(start_sample, total_samples))

        stop_sample = max(0, min(stop_sample, total_samples))

        # Retain the audio before the interval being removed
        if start_sample > previous_stop_sample:
            retained_segments.append(audio[..., previous_stop_sample:start_sample])

        # Skip through the end of the removal interval
        previous_stop_sample = max(previous_stop_sample, stop_sample)

    # Retain everything after the final removal interval
    if previous_stop_sample < total_samples:
        retained_segments.append(audio[..., previous_stop_sample:])

    # This occurs if the removal intervals cover the entire recording
    if not retained_segments:
        return audio[..., :0]

    return np.concatenate(retained_segments, axis=-1)

In [24]:
def save_audio(output_path: Path, audio: np.ndarray, sample_rate: int) -> None:
    """
    Save mono or multichannel audio with soundfile.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if audio.shape[-1] == 0:
        raise ValueError(
            "The processed audio contains no samples. "
            "The removal intervals may cover the entire recording."
        )

    # librosa uses:
    #     channels x samples
    #
    # soundfile expects:
    #     samples x channels
    if audio.ndim == 2:
        audio_to_save = audio.T
    else:
        audio_to_save = audio

    sf.write(file=output_path, data=audio_to_save, samplerate=sample_rate)

In [25]:
def find_matching_audio_file(transcript_path: Path) -> Path | None:
    """
    Find the audio file matching a transcript.

    Example
    -------
    300_TRANSCRIPT.csv -> 300_AUDIO.wav
    """
    if not transcript_path.name.endswith(
        TRANSCRIPT_SUFFIX
    ):
        return None

    filename_prefix = transcript_path.name[:-len(TRANSCRIPT_SUFFIX)]

    expected_audio_path = (
        transcript_path.parent
        / f"{filename_prefix}{AUDIO_SUFFIX}"
    )

    if expected_audio_path.exists():
        return expected_audio_path

    # Case-insensitive fallback for WAV extensions or filenames.
    candidate_audio_files = [
        path
        for path in transcript_path.parent.iterdir()
        if (
            path.is_file()
            and path.name.lower().endswith(
                AUDIO_SUFFIX.lower()
            )
        )
    ]

    if len(candidate_audio_files) == 1:
        print(
            f"Warning: exact audio match was not found for "
            f"{transcript_path.name}. Using "
            f"{candidate_audio_files[0].name}."
        )
        return candidate_audio_files[0]

    if len(candidate_audio_files) > 1:
        print(
            f"Warning: multiple candidate audio files were found for "
            f"{transcript_path.name}:"
        )

        for candidate in candidate_audio_files:
            print(f"  {candidate.name}")

    return None

In [26]:
def create_output_path(audio_path: Path, root_directory: Path, processed_directory: Path) -> Path:
    """
    Create the output path while preserving the original folder
    structure beneath the Processed directory.

    Example
    -------
    Input:
        Dataset/300/300_AUDIO.wav

    Output:
        Dataset/Processed/300/300_AUDIO_removed_interviewer.wav
    """
    relative_folder = audio_path.parent.relative_to(root_directory)

    output_filename = (
        f"{audio_path.stem}_removed_interviewer"
        f"{audio_path.suffix}"
    )

    return (
        processed_directory
        / relative_folder
        / output_filename
    )

In [27]:
def copy_transcript(transcript_path: Path, output_audio_path: Path) -> Path:
    """
    Copy the transcript into the processed folder alongside the
    processed audio.

    Returns
    -------
    Path
        Path to the copied transcript.
    """
    output_transcript_path = (
        output_audio_path.parent
        / transcript_path.name
    )

    shutil.copy2(transcript_path, output_transcript_path)

    return output_transcript_path

In [39]:
def process_recording(transcript_path: Path, audio_path: Path, output_path: Path) -> dict[str, object]:
    """
    Process one transcript/audio pair.

    The processed audio removes:

    - everything before the first valid transcript timestamp
    - every interval where Ellie speaks
    - pauses between consecutive Ellie transcript rows
    """
    intervals_to_remove = read_removal_intervals(transcript_path=transcript_path, interviewer_name=INTERVIEWER_NAME)

    intervals_to_remove = merge_removal_intervals(intervals_to_remove)

    print("Intervals being removed:")

    for start_time, stop_time in intervals_to_remove:
        print(
            f"  {start_time:.3f} to "
            f"{stop_time:.3f} seconds"
        )

    # sr=None preserves the original sample rate.
    # mono=False preserves the original channel structure.
    audio, sample_rate = librosa.load(audio_path, sr=None, mono=False)

    processed_audio = remove_audio_intervals(
        audio=audio,
        sample_rate=sample_rate,
        intervals_to_remove=intervals_to_remove,
    )

    save_audio(
        output_path=output_path,
        audio=processed_audio,
        sample_rate=sample_rate,
    )

    copied_transcript = copy_transcript(
        transcript_path=transcript_path,
        output_audio_path=output_path,
    )

    original_duration = (audio.shape[-1] / sample_rate)

    processed_duration = (processed_audio.shape[-1] / sample_rate)

    removed_duration = (original_duration - processed_duration)

    return {
        "transcript": transcript_path,
        "copied_transcript": copied_transcript,
        "audio": audio_path,
        "output": output_path,
        "number_of_intervals": len(
            intervals_to_remove
        ),
        "original_duration": original_duration,
        "processed_duration": processed_duration,
        "removed_duration": removed_duration,
    }

In [44]:
def process_dataset(root_directory: Path, processed_directory: Path, overwrite: bool = False) -> None:
    """
    Recursively find and process all transcript/audio pairs.
    """
    root_directory = root_directory.expanduser().resolve()
    processed_directory = (
        processed_directory.expanduser().resolve()
    )

    if not root_directory.exists():
        raise FileNotFoundError(
            f"Root directory does not exist: "
            f"{root_directory}"
        )

    if not root_directory.is_dir():
        raise NotADirectoryError(
            f"Root path is not a directory: "
            f"{root_directory}"
        )

    processed_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    transcript_paths = sorted(
        transcript_path
        for transcript_path in root_directory.rglob(
            f"*{TRANSCRIPT_SUFFIX}"
        )
        if (
            transcript_path.is_file()
            and not transcript_path.name.startswith("._")
            and processed_directory not in transcript_path.parents
        )
    )

    print(
        f"Found {len(transcript_paths)} transcript file(s)."
    )

    successful_count = 0
    skipped_count = 0
    failed_count = 0

    summary_rows = []

    for index, transcript_path in enumerate(transcript_paths, start=1):
        print(
            f"\n{'=' * 70}"
        )
        print(
            f"[{index}/{len(transcript_paths)}] "
            f"Processing transcript:"
        )
        print(transcript_path)

        audio_path = find_matching_audio_file(transcript_path)

        if audio_path is None:
            print(
                "Skipped: no matching audio file was found."
            )
            skipped_count += 1
            continue

        output_path = create_output_path(
            audio_path=audio_path,
            root_directory=root_directory,
            processed_directory=processed_directory,
        )

        if output_path.exists() and not overwrite:
            print(
                f"Skipped: output already exists:\n"
                f"{output_path}"
            )
            skipped_count += 1
            continue

        try:
            result = process_recording(
                transcript_path=transcript_path,
                audio_path=audio_path,
                output_path=output_path,
            )

            percent_removed = (100 * result["removed_duration"] / result["original_duration"])

            summary_rows.append(
            {
                "participant_id": transcript_path.parent.name,
                "transcript_filename": transcript_path.name,
                "audio_filename": audio_path.name,
                "processed_audio_filename": output_path.name,
                "original_duration_seconds": round(
                    result["original_duration"], 3
                ),
                "processed_duration_seconds": round(
                    result["processed_duration"], 3
                ),
                "removed_duration_seconds": round(
                    result["removed_duration"], 3
                ),
                "number_of_removal_intervals": result[
                    "number_of_intervals"
                ],
                "percent_removed": round(percent_removed, 2),
                "status": "success",
                "error": "",
            }
        )

            print(
                f"Saved:\n{result['output']}"
            )
            print(
                "Removal intervals: "
                f"{result['number_of_intervals']}"
            )
            print(
                "Original duration: "
                f"{result['original_duration']:.2f} seconds"
            )
            print(
                "Processed duration: "
                f"{result['processed_duration']:.2f} seconds"
            )
            print(
                "Removed duration: "
                f"{result['removed_duration']:.2f} seconds"
            )

            successful_count += 1

        except Exception as error:
            summary_rows.append(
            {
                "participant_id": transcript_path.parent.name,
                "transcript_filename": transcript_path.name,
                "audio_filename": (
                    audio_path.name
                    if audio_path is not None
                    else ""
                ),
                "processed_audio_filename": "",
                "original_duration_seconds": "",
                "processed_duration_seconds": "",
                "removed_duration_seconds": "",
                "number_of_removal_intervals": "",
                "percent_removed": "",
                "status": "failed",
                "error": str(error),
            }
        )
            print(
                f"Failed to process "
                f"{transcript_path.name}: {error}"
            )
            failed_count += 1

    summary_df = pd.DataFrame(summary_rows)

    summary_df.to_csv(processed_directory / "processing_summary.csv", index=False)
    print(
        f"\n{'=' * 70}"
    )
    print("Processing complete.")
    print(f"Successful: {successful_count}")
    print(f"Skipped:    {skipped_count}")
    print(f"Failed:     {failed_count}")
    print(f"Output:     {processed_directory}")

In [45]:
if __name__ == "__main__":
    process_dataset(
        root_directory=ROOT_DIRECTORY,
        processed_directory=PROCESSED_DIRECTORY,
        overwrite=OVERWRITE_EXISTING_FILES,
    )

Found 186 transcript file(s).

[1/186] Processing transcript:
/Volumes/LaCie/DAIC-WOZ/Dataset/300_P/300_TRANSCRIPT.csv
Intervals being removed:
  0.000 to 61.378 seconds
  63.798 to 67.528 seconds
  70.978 to 74.198 seconds
  82.218 to 83.578 seconds
  85.558 to 87.898 seconds
  98.228 to 100.868 seconds
  110.788 to 113.168 seconds
  119.218 to 119.978 seconds
  122.828 to 126.958 seconds
  133.718 to 136.178 seconds
  146.008 to 147.468 seconds
  155.698 to 160.858 seconds
  163.618 to 164.328 seconds
  174.148 to 177.668 seconds
  182.508 to 186.858 seconds
  193.678 to 195.248 seconds
  199.288 to 202.398 seconds
  211.648 to 216.068 seconds
  221.358 to 222.068 seconds
  223.708 to 227.208 seconds
  235.458 to 237.378 seconds
  242.218 to 243.518 seconds
  252.938 to 258.848 seconds
  262.748 to 263.938 seconds
  269.898 to 276.788 seconds
  288.858 to 293.268 seconds
  298.928 to 303.498 seconds
  314.148 to 316.388 seconds
  322.918 to 324.388 seconds
  334.268 to 335.058 second